# NPTEL Study Notes: LLM Alignment and Proximal Policy Optimization (PPO)

Welcome to your study guide for LLM Alignment. These notes bridge the gap between high-level summaries and dense mathematical proofs, specifically tailored for NPTEL exam preparation. We will cover how a Policy Model (the LLM) is mathematically optimized using a Reward Model, addressing the challenges of reward hacking, non-differentiable sampling, and training instability.

---

## 1. The Core Objective: Regularized Reward Maximization

To align an LLM with human preferences, we use two models:

1. **Policy Model ($\pi_\theta$):** The LLM being fine-tuned, parameterized by weights $\theta$.
2. **Reward Model:** A separate LLM trained on human feedback that scores the responses generated by the Policy Model.

### The Problem: Reward Hacking

If we strictly maximize the reward given by the Reward Model, the Policy Model will eventually "hack" the objective. It will exploit loopholes in the Reward Model to achieve high scores by generating unrealistic or degenerate responses (e.g., outputting repetitive text, verbose nonsense, or excessive emoticons).

### The Solution: KL Divergence Regularization

To prevent reward hacking, we restrict how much the Policy Model can change from its original state. We introduce a penalty using **Kullback-Leibler (KL) Divergence**, which measures the difference between two probability distributions.

The objective function becomes a **Regularized Reward**:


$$\text{Maximize } \mathbb{E}_{x \sim D, y \sim \pi_\theta} \left[ R(x,y) - \lambda \cdot D_{KL}(\pi_\theta(y\vert{}x) \vert{}\vert{} \pi_{ref}(y\vert{}x)) \right]$$

* $x$: The input prompt (e.g., "Where is the Taj Mahal?").
* $y$: The generated response (e.g., "The Taj Mahal is in Agra").
* $R(x,y)$: The score from the Reward Model.
* $\pi_{ref}$: The reference model (the initial, un-finetuned LLM).
* $\lambda$: A scaling factor controlling the penalty strength. Higher $\lambda$ forces the updated model to stay closer to the original model.

> **NPTEL Exam Tip:** Remember that KL Divergence acts as a *regularizer*. Its job is to ensure the output retains a reasonable probability under the original model while still maximizing the reward.

---

## 2. The Optimization Challenge: Non-Differentiable Sampling

To maximize the objective function, we need to update the model parameters $\theta$ using gradient ascent: $\theta_{new} = \theta_{old} + \alpha \nabla_\theta J(\theta)$.

However, standard (vanilla) gradient descent **fails here**.

* **Why?** The expectation $\mathbb{E}$ requires us to generate (sample) discrete text tokens from the policy model. Sampling from a discrete distribution (like picking a specific word from a vocabulary) is a **highly non-differentiable** operation. You cannot compute a standard gradient through the sampling step.

---

## 3. The REINFORCE Algorithm & The Log Derivative Trick

To bypass the non-differentiability of sampling, we use the **REINFORCE algorithm** from Reinforcement Learning. It relies on a mathematical identity known as the **Log Derivative Trick**.

### The Log Derivative Trick

From basic calculus, the derivative of $\log z$ is $\frac{1}{z} \cdot \frac{dz}{dx}$. Rearranging this for a probability distribution $\pi_\theta$:


$$\nabla_\theta \pi_\theta(y\vert{}x) = \pi_\theta(y\vert{}x) \nabla_\theta \log \pi_\theta(y\vert{}x)$$

By substituting this identity into the gradient of our expected reward, we transform the intractable gradient of an expectation into an expectation of a gradient:


$$\nabla_\theta \mathbb{E}[R] = \mathbb{E}_{y \sim \pi_\theta} \left[ R(x,y) \cdot \nabla_\theta \log \pi_\theta(y\vert{}x) \right]$$

### Monte Carlo Estimation

Because an LLM can generate an infinite number of possible responses ($y$), we cannot compute the exact expectation. Instead, we use **Monte Carlo sampling**: we generate a batch of responses, compute $R \cdot \nabla_\theta \log \pi_\theta$ for each, and take the average.

---

## 4. The Credit Assignment Problem

In basic REINFORCE, the reward $R$ is calculated for the **entire sequence** after it finishes generating.

**Example Scenario:**

* Prompt ($x$): "Where is the Taj Mahal?"
* Response ($y$): "The Taj Mahal is in France."
* Reward ($R$): Negative (because the answer is wrong).

**The flaw:** The words "The", "Taj", and "Mahal" are factually appropriate continuations of the prompt. However, because the sentence ended poorly ("in France"), the *entire sequence* gets a negative reward. This suppresses the probability of generating perfectly good tokens like "Taj". This causes high variance and unstable training.

---

## 5. Token-Level Rewards: Q-Function and Value Function

To solve the credit assignment problem, we move from sequence-level rewards to token-level rewards using Q-functions and Value functions.

### The Q-Function (State-Action Value)

The Q-function $Q(s_t, a_t)$ calculates the average **discounted cumulative reward** from a specific state-action pair onwards.

* **State ($s_t$):** The prompt plus the tokens generated so far (e.g., "Where is the Taj Mahal?" + "The").
* **Action ($a_t$):** The specific next token chosen (e.g., "Taj").

$$Q(s_t, a_t) = \mathbb{E} \left[ r_t + \gamma r_{t+1} + \gamma^2 r_{t+2} + ... \right]$$

* $\gamma$ **(Discount Factor, $0 \le \gamma \le 1$):** Reduces the weight of future tokens. The word "France" at the end of the sentence has less impact on the reward for the word "Taj" at the beginning of the sentence.

### The Value Function ($V(s_t)$)

The Value function $V(s_t)$ predicts the average discounted cumulative reward from a state $s_t$, *regardless of the specific action taken*. It acts as a baseline of how "good" a state is.

---

## 6. The Advantage Function

Empirically, using just the Q-function is not stable enough. We introduce the **Advantage Function $A(s_t, a_t)$**:


$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$$

* **Intuition:** The Advantage function measures how much *better* a specific action (token) is compared to the *average* action taken from that state.
* If $A > 0$, the token was better than average (increase its probability).
* If $A < 0$, the token was worse than average (decrease its probability).

In our final REINFORCE algorithm, we replace the overall reward $R$ with the Advantage function $A(s_t, a_t)$ for every token.

---

## 7. Off-Policy Learning & Importance Weights

**The Inefficiency:** In standard REINFORCE, after every single parameter update ($\theta$), you must generate entirely new responses to calculate the next gradients. Generating text from an LLM is computationally expensive.

**The Fix:** We generate a large batch of responses using an *older* version of the model ($\pi_{\theta_{old}}$). We evaluate the new model ($\pi_\theta$) using these pre-generated responses by applying **Importance Weights**:


$$\text{Importance Weight} = \frac{\pi_\theta(a_t\vert{}s_t)}{\pi_{\theta_{old}}(a_t\vert{}s_t)}$$


This ratio corrects for the fact that the data was generated by a different, older policy.

---

## 8. Proximal Policy Optimization (PPO) Clipping

While Importance Weights solve the efficiency problem, they introduce a new instability. If the new model diverges too much from the old model, the Importance Weight ratio can explode (become drastically large or small), ruining the gradient update.

**PPO (Proximal Policy Optimization)** fixes this by **clipping** the importance weight ratio so it stays within a strict bound, typically $[1-\epsilon, 1+\epsilon]$ (where $\epsilon$ is a hyperparameter, e.g., 0.2).

* If the Advantage is **positive** (good token), we cap the ratio at $1+\epsilon$. (Don't update too aggressively in the positive direction).
* If the Advantage is **negative** (bad token), we floor the ratio at $1-\epsilon$ (using a $\max$ operation). (Don't penalize too aggressively).

---

## 9. Neural Network Implementation of the Value Function

How does the LLM compute $V(s_t)$?

1. **Architecture:** We add a simple **Linear Layer** on top of the LLM's final token embeddings. This layer (parameterized by $\phi$) predicts a single scalar value: $V_\phi(s_t)$.
2. **Training:** We train this linear layer using **Mean Squared Error (MSE)**. We minimize the difference between the predicted value $V_\phi(s_t)$ and the actual *Reward-to-Go* (the actual discounted cumulative reward calculated after the sequence finishes).

$$\text{Loss}_{Value} = \sum \left( V_\phi(s_t) - R_{to\_go} \right)^2$$



---

## Practice Questions

**Q1. In RLHF, what is the primary consequence of maximizing the reward model's score without any regularization?**
A) The model parameters fail to update.
B) The model suffers from catastrophic forgetting of grammatical rules.
C) The policy model "hacks" the reward model, producing unrealistic or degenerate responses that exploit the scoring mechanism.
D) The KL divergence goes to zero.

* **Answer:** C. Without a KL divergence penalty to keep the model anchored to the reference model, it will generate verbose or nonsensical text (like endless emoticons) to hack the reward function.

**Q2. Why is vanilla gradient descent inadequate for optimizing the policy model directly?**
A) The learning rate is too small for Large Language Models.
B) The process of sampling discrete tokens from the output distribution is non-differentiable.
C) The reward model outputs discrete integers instead of continuous floats.
D) Vanilla gradient descent cannot minimize KL divergence.

* **Answer:** B. You cannot backpropagate through the discrete sampling step. The log-derivative trick in the REINFORCE algorithm is required to reframe this as a differentiable expectation.

**Q3. If a policy model generates the sequence $s = [a_1, a_2, a_3, a_4]$, and the individual rewards at each step are $r_1=0, r_2=1, r_3=0, r_4=-5$. Assuming a discount factor $\gamma = 0.9$, what is the Q-function (cumulative discounted reward) starting from step $t=2$?**
A) -4.0
B) 1.0 + 0 + (-5.0) = -4.0
C) 1.0 + 0.9(0) + 0.81(-5.0) = -3.05
D) 0.9(1) + 0.81(0) + 0.729(-5.0) = -2.745

* **Answer:** C. $Q(s_2, a_2) = r_2 + \gamma r_3 + \gamma^2 r_4 = 1 + 0.9(0) + (0.9)^2(-5) = 1 + 0 - 4.05 = -3.05$.

**Q4. What is the mathematical definition of the Advantage function $A(s_t, a_t)$ in the context of LLM alignment?**
A) $A(s_t, a_t) = Q(s_t, a_t) + V(s_t)$
B) $A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$
C) $A(s_t, a_t) = V(s_t) - Q(s_t, a_t)$
D) $A(s_t, a_t) = R_{cumulative} \times \gamma$

* **Answer:** B. The advantage function is the state-action value (Q) minus the average state value (V). It represents how much better an action is relative to the baseline.

**Q5. What is the exact purpose of the "clip" operation in Proximal Policy Optimization (PPO)?**
A) To limit the maximum length (number of tokens) of the generated response.
B) To clip the gradients of the linear Value Function layer to prevent exploding gradients.
C) To bound the Importance Weight ratio between $1-\epsilon$ and $1+\epsilon$, ensuring stable training updates and preventing drastically large policy shifts.
D) To limit the maximum possible reward to 1.0.

* **Answer:** C. PPO clipping prevents the old policy and new policy from diverging too rapidly during off-policy updates.

---

## Quick Revision Summary

* **Objective:** Maximize expected Reward minus KL Divergence (to prevent Reward Hacking).
* **Log-Derivative Trick:** $\nabla \pi = \pi \nabla \log \pi$. Converts non-differentiable sampling into an expectation we can approximate with Monte Carlo sampling (REINFORCE).
* **Credit Assignment:** Sequence-level reward wrongly penalizes good early tokens for bad later tokens.
* **Discounted Rewards ($\gamma$):** Weighs immediate tokens heavier than distant future tokens.
* **Advantage Function ($Q - V$):** Measures token performance against the baseline. Replaces flat reward for stability.
* **Importance Weights:** $\pi_{new}/\pi_{old}$. Allows reusing old sampled responses to update new weights (saves compute).
* **PPO Clipping:** Bounds importance weights to $[1-\epsilon, 1+\epsilon]$ to guarantee stable, measured policy updates.
* **Value Network:** A linear layer added to token embeddings, trained via Mean Squared Error (MSE) against the actual cumulative reward.